In [ ]:
import requests
import json
import base64

from vantage6.client import UserClient

In [ ]:
# This is standard authentication Keycloak flow. @Itziar; we need to discuss on how
# to deal with this for the demo. We could create a token that is valid for 10 years
# and use that token to authenticate?
client = UserClient(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es:443/server",
    auth_url="https://vantage6-auth.orchestrator.idea.lst.tfo.upm.es:443",
    auth_client="public_client",
    auth_realm="vantage6",
    log_level="INFO"
)
# You can authenticate using the user `itziar` and the password that I've send to you.
client.authenticate()

# Set the headers for the other requests
headers = {
    "Authorization": f"Bearer {client._access_token}"
}

# Print the server version
print("Server version: ", client.util.get_server_version())

In [ ]:
#
# Static content
#
image = "harbor2.vantage6.ai/idea4rc/sessions:latest"
method = "create_cohort"

# Organization IDs for the test collaboration with FAKE OMOP data
UPM_ORG_ID = 3
IKNL_ORG_ID = 1
ORG_IDS = [UPM_ORG_ID, IKNL_ORG_ID]

# Collaboration ID for the test collaboration with FAKE OMOP data
COLLABORATION_ID = 2

# Demo session ID
SESSION_ID = 2

# All organizations in the demo collaboration
STUDY_ID = 3

DATAFRAME_ID_PELVIS = 76 # Pelvis
DATAFRAME_ID_RPS_PELVIS = 77 # RPS+Pelvis
DATAFRAME_ID_RPS = 78 # RPS

#
# Dynamic content
#
RESULTS_COL = "sex"
GROUP_COLS = ["fnclcc_grade"]


org_input = [
    {
        "id": UPM_ORG_ID, # Central task is executed by UPM
        "arguments": base64.b64encode(
            json.dumps(
                {
                    "results_col": RESULTS_COL,
                    "group_cols": GROUP_COLS,
                    "organizations_to_include": ORG_IDS
                }
            ).encode("UTF-8")
        ).decode("UTF-8")
    }
]

In [ ]:
payload = {
    "name": "Human-readable name of the task",
    "image": "harbor2.vantage6.ai/idea4rc/analytics:latest",
    "description": "Description of the task",
    "action": "central_compute",
    "method": "crosstab",
    "organizations": org_input,
    "databases": [
        [
            {
                "type": "dataframe",
                "dataframe_id": DATAFRAME_ID_RPS
            },
            {
                "type": "dataframe",
                "dataframe_id": DATAFRAME_ID_RPS_PELVIS
            },
            {
                "type": "dataframe",
                "dataframe_id": DATAFRAME_ID_PELVIS
            }
        ]
    ],
    "session_id": SESSION_ID,
    "study_id": STUDY_ID
}

In [ ]:
# Create a vantage6 task to execute the summary analysis.
response = requests.post(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/task",
    headers=headers,
    json=payload
)
TASK_ID = response.json()["id"]
JOB_ID = response.json()["job_id"]
response.json()

In [ ]:
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/run?job_id={JOB_ID}",
    headers=headers,
)
response.json()